# 🧭 Anchor Policy Ablation — Quick Tour

This notebook is the **policy-lab entry point** for the new Stage 2.6.

It does **one thing only**: compare anchoring policies on top of a single Stage-2 load, pick the best policy **using validation only**, and write a small replay artifact for later use in the main ensemble notebook.

## What it does

**Stage 2 once → replay many policies → compare wells × policies → select by validation → save `selected_anchor_policies.csv`**

## Frozen methodological rule

- **Policy is selected on validation**
- **Test is audit only**
- On **test**, the **policy stays fixed**, but the **anchor is recomputed from causal history available at test time**
- This is **not tuning on test**; it is **causal replay under a fixed rule**

## What you get

- `comparison_df`: one row per **well × policy**
- `selected_df`: chosen policy per well
- `anchor_policy_comparison.csv`
- `selected_anchor_policies.csv`

## When to use this notebook

Use this notebook to **explore and choose anchoring policies**.

Use your existing **ensemble analysis notebook** later to **replay, visualize, and audit** the chosen policy artifact inside the normal Phase 4 flow.

In [ ]:
# %% Anchor Policy Ablation — single-cell entry point
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from common.phase_orchestrator import Phase4Config
from postproc_bias.ablation import run_anchor_policy_ablation
from postproc_bias.anchoring import (
    build_anchor_artifact_status,
    build_anchor_registry_table,
    build_anchor_run_summary,
    render_anchor_editorial_report,
)

# -----------------------------------------------------------------------------
# User knobs
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("/home/gabriel/Documentos/Equinor")
SERIES_STORE_ROOT = PROJECT_ROOT / "series_store"
SAVE_DIR = PROJECT_ROOT / "artifacts" / "anchor_ablation_dev"

CAMPAIGN = "HPO_153_Lag_100_Horizon_150"

POLICY_NAMES = [
    "baseline_default",
    "short_window_spike_sensitive",
    "long_window_smoother",
    "trend_strict",
    "log_space_robust",
]

WELLS_TO_ANALYZE = [
    "P11", "P12", "P13", "P14", "P15", "P16",
    "15/9-F-12", "15/9-F-14",
]

ENABLE_PLOTS_DURING_ABLATION = False
ENABLE_RISK_DURING_ABLATION = False
LOG_MODE = "compact"

SELECTION_METRIC = "val_metric_inter"
TIE_BREAK_METRIC = "val_risk_q90"

# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------
cfg = Phase4Config()
cfg.scoring_strategy = "weighted_score"
cfg.campaigns_to_ensemble = {"T_current": CAMPAIGN}
cfg.n_champions_per_group = 2
cfg.metric_weights = {
    "val_smape_agg": 5.0,
    "val_smape_cum": 1.0,
}
cfg.lower_is_better = {
    "val_smape_agg": True,
    "val_smape_cum": True,
    "weighted_score": True,
    "robust_score": True,
}
cfg.posthoc_overrides = dict(
    top_strategies_per_well=15,
    per_strategy_k=50,
    selection_strategy="best_of_the_best",
    apply_pareto=False,
    mad_guard={
        "enabled": True,
        "alpha": 0.5,
        "metrics": ["val_smape_cum", "val_smape_agg"],
        "log": True,
        "side": "right",
    },
    valcum_gate={"q_low": 0.1, "q_high": 0.9},
)
cfg.wells_to_analyze = WELLS_TO_ANALYZE
cfg.project_root = PROJECT_ROOT
cfg.series_store_root = SERIES_STORE_ROOT
cfg.palette = "default"
cfg.enable_plots = ENABLE_PLOTS_DURING_ABLATION
cfg.show_family_traces = False
cfg.show_champion_traces = False
cfg.log_mode = LOG_MODE
cfg.log_width = 100
cfg.enable_risk_plots = ENABLE_RISK_DURING_ABLATION
cfg.risk_splits = ["val+test"]
cfg.risk_horizons_days = [-1]
cfg.risk_weighting = "uniform"
cfg.risk_distance_temp = 0.5
cfg.risk_palette = "default"
cfg.risk_show_tables = True

# -----------------------------------------------------------------------------
# Run
# -----------------------------------------------------------------------------
out = run_anchor_policy_ablation(
    cfg=cfg,
    policy_names=POLICY_NAMES,
    save_dir=SAVE_DIR,
    selection_metric=SELECTION_METRIC,
    tie_break_metric=TIE_BREAK_METRIC,
)

comparison_df = out["comparison_df"]
selected_df = out["selected_df"]

# -----------------------------------------------------------------------------
# Build notebook views
# -----------------------------------------------------------------------------
pd.set_option("display.max_columns", 50)

registry_df = build_anchor_registry_table()
summary_df = build_anchor_run_summary(comparison_df, selected_df)
artifact_status_df = build_anchor_artifact_status(SAVE_DIR)

# -----------------------------------------------------------------------------
# Present
# -----------------------------------------------------------------------------
display(Markdown("## Policies"))
display(registry_df)

display(Markdown("## Run summary"))
display(summary_df)

editorial = render_anchor_editorial_report(
    comparison_df=comparison_df,
    selected_df=selected_df,
    include_risk_if_available=False,
    show_full_comparison=False,
)

display(Markdown("## Saved artifacts"))
display(artifact_status_df)